# Lesson 8 - Creating an A2A Healthcare Provider Agent using LangGraph and MCP

In [1]:
%cd /content/drive/MyDrive/A2A-MCP

/content/drive/MyDrive/A2A-MCP


In [2]:
%%writefile requirements.txt
google-adk[a2a]==1.19.0
a2a-sdk[http-server]==0.3.16
mcp==1.19.0
agent-framework==1.0.0b251120
agent-framework-a2a==1.0.0b251120
langchain-mcp-adapters
langgraph
langchain[openai]
google-auth==2.42.0
httpx==0.28.1
ipython==8.32.0
nest-asyncio==1.6.0
python-dotenv==1.0.1
requests>=2.32.4,<3.0.0
uvicorn==0.34.0

Overwriting requirements.txt


In this lesson, you will build a third agent: a Healthcare Provider Agent. This agent demonstrates a powerful combination of technologies:
1.  **MCP (Model Context Protocol)**: You will build a server that exposes a tool to find doctors from a JSON file.
2.  **LangGraph**: You will build an agent that uses this MCP tool.
3.  **A2A**: You will wrap the LangGraph agent in an A2A server.

## 8.1. Create the MCP Server

First, you will define an MCP server using `FastMCP`. This server exposes a tool called `list_doctors` which queries a local `doctors.json` file.

In [3]:
%%writefile mcpserver.py
import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

# Initialize the server
mcp = FastMCP("doctorserver")

# Load Data
doctors: list = json.loads(Path("data/doctors.json").read_text())


@mcp.tool()
def list_doctors(state: str | None = None, city: str | None = None) -> list[dict]:
    """This tool returns a list of doctors practicing in a specific location. The search is case-insensitive.

    Args:
        state: The two-letter state code (e.g., "CA" for California).
        city: The name of the city or town (e.g., "Boston").

    Returns:
        A JSON string representing a list of doctors matching the criteria.
        If no criteria are provided, an error message is returned.
        Example: '[{"name": "Dr John James", "specialty": "Cardiology", ...}]'
    """
    # Input validation: ensure at least one search term is given.
    if not state and not city:
        return [{"error": "Please provide a state or a city."}]

    target_state = state.strip().lower() if state else None
    target_city = city.strip().lower() if city else None

    return [
        doc
        for doc in doctors
        if (not target_state or doc["address"]["state"].lower() == target_state)
        and (not target_city or doc["address"]["city"].lower() == target_city)
    ]


# Kick off server if file is run
if __name__ == "__main__":
    mcp.run(transport="stdio")


Overwriting mcpserver.py


In [4]:
import asyncio
from IPython.display import Markdown, display
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.sessions import StdioConnection
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

from helpers import setup_env, get_gemini_api_key, get_gemini_model, get_gemini_openai_base_url

In [5]:
setup_env()

In [6]:
gemini_api_key = get_gemini_api_key()
model = get_gemini_model()
base_url = get_gemini_openai_base_url()

In [7]:
import nest_asyncio
nest_asyncio.apply()

## 8.3. Define the Provider Agent Class

This class handles the MCP client connection and agent initialization.

PS: connect to the MCP server and use it within a LangChain/LangGraph agent. You use `MultiServerMCPClient` to connect to the local script via `stdio` transport.


In [8]:
%%writefile -a agents.py

class ProviderAgent:
    def __init__(self) -> None:
        setup_env()
        # Using Gemini API key with the OpenAI-compatible endpoint
        self.gemini_api_key = get_gemini_api_key()
        self.model = get_gemini_model()
        self.base_url = get_gemini_openai_base_url()

        self.mcp_client = MultiServerMCPClient(
            {
                "find_healthcare_providers": StdioConnection(
                    transport="stdio",
                    command="uv",
                    args=["run", "mcpserver.py"],
                )
            }
        )

        self.agent = None

    async def initialize(self):
        """Initialize the agent asynchronously."""
        tools = await self.mcp_client.get_tools()
        self.agent = create_react_agent(
            # We use ChatOpenAI as a client for the Gemini OpenAI-compatible API
            ChatOpenAI(
                model=self.model,
                openai_api_key=self.gemini_api_key,
                openai_api_base=self.base_url,
            ),
            tools,
            state_modifier="""Your task is to find and list providers using 
            the find_healthcare_providers MCP Tool based on the users query. 
            Only use providers based on the response from the tool. Output 
            the information in a table.""",
        )
        return self

    async def answer_query(self, prompt: str) -> str:
        if self.agent is None:
            raise RuntimeError(
                "Agent not initialized. Call initialize() first."
            )

        response = await self.agent.ainvoke(
            {
                "messages": [
                    {
                        "role": "user",
                        "content": prompt,
                    }
                ]
            }
        )
        return response["messages"][-1].content


Appending to agents.py


## 8.4. Wrap in A2A Server

Finally, create the `a2a_provider_agent.py` server file. This uses the standard A2A wrapping pattern you learned in Lesson 3, but with the added complexity of asynchronous initialization for the MCP client in the `ProviderAgentExecutor`.

In [9]:
%%writefile a2a_provider_agent.py
from dotenv import load_dotenv
import os
import uvicorn
from a2a.server.agent_execution import AgentExecutor, RequestContext
from a2a.server.apps import A2AStarletteApplication
from a2a.server.events import EventQueue
from a2a.server.request_handlers import DefaultRequestHandler
from a2a.server.tasks import InMemoryTaskStore
from a2a.types import (
    AgentCapabilities,
    AgentCard,
    AgentSkill,
)
from a2a.utils import new_agent_text_message
from agents import ProviderAgent

class ProviderAgentExecutor(AgentExecutor):
    """This is an agent for finding healthcare providers based on location and specialty."""
    
    def __init__(self) -> None:
        # Don't await in __init__ - it's not async
        self.agent = None
    
    async def _ensure_initialized(self) -> None:
        """Lazy initialization of the agent."""
        if self.agent is None:
            self.agent = await ProviderAgent().initialize()
    
    async def execute(
        self,
        context: RequestContext,
        event_queue: EventQueue,
    ) -> None:
        await self._ensure_initialized()
        
        prompt = context.get_user_input()
        response = await self.agent.answer_query(prompt)
        await event_queue.enqueue_event(new_agent_text_message(response))
    
    async def cancel(self, context: RequestContext, event_queue: EventQueue) -> None:
        pass

def main():
    print("Running Healthcare Provider Agent")
    load_dotenv()
    
    HOST = os.environ.get("AGENT_HOST", "localhost")
    PORT = int(os.environ.get("PROVIDER_AGENT_PORT", 9997))
    
    skill = AgentSkill(
        id="find_healthcare_providers",
        name="Find Healthcare Providers",
        description="Finds and lists healthcare providers based on user's location and specialty.",
        tags=["healthcare", "providers", "doctor", "psychiatrist"],
        examples=[
            "Are there any Psychiatrists near me in Boston, MA?",
            "Find a pediatrician in Springfield, IL.",
        ],
    )
    
    agent_card = AgentCard(
        name="HealthcareProviderAgent",
        description="An agent that can find and list healthcare providers based on a user's location and desired specialty.",
        url=f"http://{HOST}:{PORT}/",
        version="1.0.0",
        default_input_modes=["text"],
        default_output_modes=["text"],
        capabilities=AgentCapabilities(streaming=False),
        skills=[skill],
    )
    
    request_handler = DefaultRequestHandler(
        agent_executor=ProviderAgentExecutor(),
        task_store=InMemoryTaskStore(),
    )
    
    server = A2AStarletteApplication(
        agent_card=agent_card,
        http_handler=request_handler,
    )
    
    uvicorn.run(server.build(), host=HOST, port=PORT)
    
if __name__ == "__main__":
    main()

Overwriting a2a_provider_agent.py


## 8.6. Run the Provider Server

Activate the Provider Agent in Terminal 3:
- Open Terminal 3 by running the cell below.
- Type `uv run a2a_provider_agent.py`

In [ ]:
!uv run a2a_provider_agent.py

Running Healthcare Provider Agent
INFO:     Started server process [30173]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:9997 (Press CTRL+C to quit)


## 8.7. Resources

- [Model Context Protocol (MCP)](https://modelcontextprotocol.io/)
- [LangChain MCP Adapters](https://docs.langchain.com/oss/python/langchain/mcp)
- [Equivalent notebook in the course repo](https://github.com/holtskinner/A2AWalkthrough/blob/main/6_A2AxMCPLangGraph.ipynb)
